In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer

OUTPUT_ROOT = Path("../../outputs")
EVALUATION_DIR = OUTPUT_ROOT / "model_evaluation"
RESULT_DIR = OUTPUT_ROOT / "recalibration"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
EPSILON = 1e-6

def logit(probability):
    probability = np.clip(np.asarray(probability, float), EPSILON, 1 - EPSILON)
    return np.log(probability / (1 - probability))

def expit(value):
    value = np.asarray(value, float)
    return np.where(value >= 0, 1 / (1 + np.exp(-value)), np.exp(value) / (1 + np.exp(value)))

def fit_logistic_recalibration(y_true, oof_probability):
    x = logit(oof_probability)
    y_true = np.asarray(y_true, float)
    def negative_log_likelihood(parameters):
        linear_predictor = parameters[0] + parameters[1] * x
        return np.sum(np.logaddexp(0, linear_predictor) - y_true * linear_predictor)
    fitted = minimize(negative_log_likelihood, [0.0, 1.0], method="BFGS")
    if not fitted.success or fitted.x[1] <= 0:
        raise RuntimeError(f"Invalid logistic recalibration fit: {fitted.message}")
    return float(fitted.x[0]), float(fitted.x[1])

def calibration_metrics(y_true, probability):
    x = logit(probability)
    def negative_log_likelihood(parameters):
        linear_predictor = parameters[0] + parameters[1] * x
        return np.sum(np.logaddexp(0, linear_predictor) - y_true * linear_predictor)
    fitted = minimize(negative_log_likelihood, [0.0, 1.0], method="BFGS")
    smoother = make_pipeline(SplineTransformer(n_knots=5, degree=3, include_bias=False), LogisticRegression(C=1e6, max_iter=3000))
    smoother.fit(x.reshape(-1, 1), y_true)
    observed = smoother.predict_proba(x.reshape(-1, 1))[:, 1]
    error = np.abs(observed - probability)
    return float(fitted.x[0]), float(fitted.x[1]), float(error.mean()), float(np.median(error)), float(np.quantile(error, 0.9)), smoother

manifest = pd.read_csv(EVALUATION_DIR / "recalibration_input_manifest.csv")
rows = []
curves = {}
for item in manifest.itertuples(index=False):
    development = pd.read_csv(Path(item.development_oof_predictions))
    validation = pd.read_csv(Path(item.validation_predictions))
    alpha, beta = fit_logistic_recalibration(development["y_true"].to_numpy(int), development["y_prob"].to_numpy(float))
    y_true = validation["y_true"].to_numpy(int)
    original = validation["y_prob"].to_numpy(float)
    recalibrated = expit(alpha + beta * logit(original))
    for status, probability in (("Original", original), ("Recalibrated", recalibrated)):
        intercept, slope, ici, e50, e90, smoother = calibration_metrics(y_true, probability)
        rows.append({
            "Model_set": item.model_set,
            "Cohort": item.cohort,
            "Outcome": item.outcome,
            "Calibration_status": status,
            "AUROC": roc_auc_score(y_true, probability),
            "AUPRC": average_precision_score(y_true, probability),
            "Brier": brier_score_loss(y_true, probability),
            "Calibration_intercept": intercept,
            "Calibration_slope": slope,
            "ICI": ici,
            "E50": e50,
            "E90": e90,
            "Recalibration_alpha": alpha,
            "Recalibration_beta": beta,
        })
        curves[(item.model_set, item.cohort, item.outcome, status)] = (probability, smoother)

pd.DataFrame(rows).to_csv(RESULT_DIR / "recalibration_performance.csv", index=False)

for model_set in manifest["model_set"].drop_duplicates():
    for cohort in manifest.loc[manifest["model_set"] == model_set, "cohort"].drop_duplicates():
        outcomes = list(manifest.loc[(manifest["model_set"] == model_set) & (manifest["cohort"] == cohort), "outcome"].drop_duplicates())
        figure, axes = plt.subplots(2, 3, figsize=(15, 9.5))
        for axis, outcome in zip(axes.ravel(), outcomes):
            maximum = 0.0
            for status, color in (("Original", "#6A6A6A"), ("Recalibrated", "#0072B2")):
                probability, smoother = curves[(model_set, cohort, outcome, status)]
                upper = min(1.0, max(float(np.quantile(probability, 0.995)), 0.02))
                grid = np.linspace(EPSILON, upper, 250)
                observed = smoother.predict_proba(logit(grid).reshape(-1, 1))[:, 1]
                axis.plot(grid, observed, color=color, label=status)
                maximum = max(maximum, upper, float(np.quantile(observed, 0.995)))
            maximum = min(1.0, maximum * 1.05)
            axis.plot([0, maximum], [0, maximum], "--", color="black", label="Ideal")
            axis.set(xlabel="Predicted probability", ylabel="Smoothed observed probability", title=outcome, xlim=(0, maximum), ylim=(0, maximum))
            axis.grid(alpha=0.2)
        axes.ravel()[-1].axis("off")
        handles, labels = axes.ravel()[0].get_legend_handles_labels()
        figure.legend(handles, labels, loc="lower right", frameon=False)
        figure.tight_layout()
        figure.savefig(RESULT_DIR / f"recalibration_{model_set}_{cohort}.png", dpi=300, bbox_inches="tight")
        plt.close(figure)
